# 01 — Transform OTEL Traces → ChatML SFT Dataset

**Purpose:** Turn raw OTEL/MLflow traces into supervised fine-tuning examples in **ChatML** format (`{"messages": [...]}`), keeping only high-quality traces, and write the result as both a **Delta table** and **JSONL files on a Volume** for downstream fine-tuning notebooks.

---

## In production: read from MLflow traces

This notebook reads the synthetic Delta table produced by `00`. **In a real pipeline you would instead read directly from MLflow Tracing**, which returns a DataFrame with one row per trace:

```python
import mlflow
# All traces logged by an experiment / production endpoint:
traces_df = mlflow.search_traces(
    experiment_ids=["<your_experiment_id>"],
    filter_string="attributes.status = 'OK'",
    max_results=10_000,
)
# traces_df has request/response columns you can map to ChatML the same way below.
```

The rest of this notebook is written so the mapping logic is identical whether the rows come from `mlflow.search_traces` or the Delta table.

In [ ]:
# No extra libraries needed — pure Spark + Python.
# (In production the mlflow.search_traces path needs: %pip install mlflow)

In [ ]:
# ─────────────────────────────────────────────────────────────
# CONFIGURATION  (must match 00)
# ─────────────────────────────────────────────────────────────
CATALOG      = "main"
SCHEMA       = "otel_finetuning"
TRACES_TABLE = f"{CATALOG}.{SCHEMA}.synthetic_traces"

CHATML_TABLE      = f"{CATALOG}.{SCHEMA}.chatml_dataset"     # Delta output (Ray reads this)
VOLUME            = f"/Volumes/{CATALOG}/{SCHEMA}/finetune"  # UC Volume for JSONL
CHATML_TRAIN_PATH = f"{VOLUME}/train.jsonl"
CHATML_EVAL_PATH  = f"{VOLUME}/eval.jsonl"

EVAL_FRACTION = 0.10
SEED          = 42

spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.finetune")

## The trace → ChatML mapping

`trace_to_chatml` is a **pure function**: given one trace's `status` and parsed `attributes`, it returns a ChatML example or `None` (dropped). Keeping it pure means we can unit-test it without Spark. A trace is dropped when it errored, has empty input/output, or is missing an assistant reply — this is the inline quality filter.

In [ ]:
import json

def trace_to_chatml(status, attributes):
    """Map one trace to {'messages': [...]} or None if it should be dropped."""
    if status != "OK":
        return None

    inputs = attributes.get("mlflow.spanInputs") or {}
    outputs = attributes.get("mlflow.spanOutputs") or {}

    in_messages = inputs.get("messages") or []
    if not in_messages:
        return None

    # Pull the assistant reply out of an OpenAI-style choices payload.
    choices = outputs.get("choices") or []
    if not choices:
        return None
    assistant = choices[0].get("message") or {}
    content = (assistant.get("content") or "").strip()
    if not content:
        return None

    # Keep the original system/user turns, append the assistant answer.
    messages = [
        {"role": m["role"], "content": m["content"]}
        for m in in_messages
        if m.get("role") in ("system", "user") and m.get("content")
    ]
    if not any(m["role"] == "user" for m in messages):
        return None

    messages.append({"role": "assistant", "content": content})
    return {"messages": messages}

### Quick local sanity check on the mapping (no Spark)

In [ ]:
# Two hand-built rows: one good, one errored.
_good = {"mlflow.spanInputs": {"messages": [
            {"role": "system", "content": "sys"},
            {"role": "user", "content": "hi"}]},
         "mlflow.spanOutputs": {"choices": [{"message": {"role": "assistant", "content": "hello"}}]}}
_bad = {"mlflow.spanInputs": {"messages": []}, "mlflow.spanOutputs": None}

assert trace_to_chatml("OK", _good)["messages"][-1] == {"role": "assistant", "content": "hello"}
assert trace_to_chatml("ERROR", _good) is None
assert trace_to_chatml("OK", _bad) is None
print("mapping logic OK")

## Apply to all traces

In [ ]:
from pyspark.sql import functions as F

raw = spark.table(TRACES_TABLE)
print(f"Read {raw.count()} raw traces")

# Parse + map on the driver via toLocalIterator would not scale; use an RDD map instead.
def row_to_example(row):
    attrs = json.loads(row["attributes"])
    return trace_to_chatml(row["status"], attrs)

examples = [e for e in (row_to_example(r) for r in raw.select("status", "attributes").collect())
            if e is not None]

print(f"Kept {len(examples)} / {raw.count()} traces after filtering")
print(json.dumps(examples[0], indent=2)[:500])

> **Scaling note:** `.collect()` is fine for a demo-sized dataset. For millions of traces, wrap `trace_to_chatml` in a Spark UDF (or `mapInPandas`) so parsing runs on the executors instead of the driver.

## Train / eval split

In [ ]:
import random

rng = random.Random(SEED)
shuffled = examples[:]
rng.shuffle(shuffled)

n_eval = max(1, int(len(shuffled) * EVAL_FRACTION))
eval_examples  = shuffled[:n_eval]
train_examples = shuffled[n_eval:]
print(f"train={len(train_examples)}  eval={len(eval_examples)}")

## Write outputs: Delta table + JSONL on Volume

In [ ]:
# --- Delta table (consumed by the Ray Train notebook 02b) ---
def to_rows(exs, split):
    return [{"split": split, "messages_json": json.dumps(e["messages"])} for e in exs]

chatml_df = spark.createDataFrame(to_rows(train_examples, "train") +
                                  to_rows(eval_examples, "eval"))
chatml_df.write.mode("overwrite").saveAsTable(CHATML_TABLE)
print(f"Wrote {chatml_df.count()} rows to {CHATML_TABLE}")

# --- JSONL files on Volume (consumed by TRL 02a and Mosaic AI 02c) ---
def write_jsonl(path, exs):
    with open(path, "w") as f:
        for e in exs:
            f.write(json.dumps(e) + "\n")

write_jsonl(CHATML_TRAIN_PATH, train_examples)
write_jsonl(CHATML_EVAL_PATH, eval_examples)
print(f"Wrote {CHATML_TRAIN_PATH} and {CHATML_EVAL_PATH}")

display(spark.table(CHATML_TABLE).limit(5))